<a href="https://colab.research.google.com/github/AufanT/AufanT-BigData26_B_2411532011_AufanTaufiqurrahman/blob/main/Praktikum_2/BD_B_P02_2411532011_AufanTaufiqurrahman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install faker

In [ ]:
import numpy as np
import pandas as pd
from faker import Faker
import random

In [ ]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + "  "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])  # rating opsional

    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


In [ ]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


In [ ]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")
print("Jumlah baris setelah drop_duplicates():", len(df))

Jumlah baris setelah drop_duplicates(): 495


In [ ]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df["transaction_id"].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


In [ ]:
# a. Standardisasi teks kategorikal (category, payment_method, shipping_city)
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

# b. Koreksi tipe data pada kolom price (dari teks bercampur simbol, menjadi numerik)
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

# c. Standardisasi format tanggal ke YYYY-MM-DD
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

# d. Finalisasi tipe data
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

In [ ]:
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

folder_drive = "/content/drive/MyDrive/BigData/Praktikum2"
os.makedirs(folder_drive, exist_ok=True)   # buat folder jika belum ada

path_drive = folder_drive + "/transaksi_bersih.csv"
df.to_csv(path_drive, index=False)

# Verifikasi: baca ulang file dari Drive
cek = pd.read_csv(path_drive)
print("Tersimpan di:", path_drive)
print("Jumlah baris:", len(cek))
print("Kolom:", list(cek.columns))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Tersimpan di: /content/drive/MyDrive/BigData/Praktikum2/transaksi_bersih.csv
Jumlah baris: 490
Kolom: ['transaction_id', 'customer_name', 'product_name', 'category', 'price', 'quantity', 'payment_method', 'transaction_date', 'shipping_city', 'rating']


## Studi Kasus

**1. Mengapa angka tim IT (515) dan tim Finance (490) berbeda?**

Angka 515 adalah jumlah baris **data mentah**, sedangkan 490 adalah jumlah baris setelah **pra-pemrosesan**. Selisih 25 baris berasal dari dua langkah:
- **20 baris** dibuang karena `customer_name` atau `payment_method` kosong. Kedua kolom ini wajib untuk identifikasi dan pencatatan transaksi.
- **5 baris** dibuang karena merupakan duplikat, yaitu transaksi yang sama tercatat dua kali.

Dengan kata lain, 515 baris mentah sebenarnya berisi 500 transaksi unik ditambah 15 salinan duplikat. Dari 500 transaksi unik tersebut, 10 transaksi tidak memiliki data wajib (termasuk salinannya), sehingga tersisa 490 transaksi valid.

**2. Apakah 490 baris "lebih benar" daripada 515?**

Untuk analisis penjualan, **ya, 490 lebih dapat dipercaya** (Veracity lebih tinggi). Dengan 515 baris, transaksi duplikat akan terhitung dua kali, sehingga total pendapatan dan jumlah transaksi menjadi *overstated*.

Namun, 490 **bukan kebenaran mutlak**:
- 10 transaksi yang dibuang karena data wajib kosong kemungkinan tetap benar-benar terjadi, hanya pencatatannya tidak lengkap. Untuk kebutuhan Finance, transaksi tersebut sebaiknya ditandai (*flag*) dan ditelusuri ke sistem sumber, bukan sekadar dihapus, agar pendapatan tidak *understated*.
- Data mentah 515 baris tetap perlu disimpan sebagai jejak audit, supaya setiap selisih bisa ditelusuri asalnya.

**3. Bagaimana menjelaskan rating yang dibiarkan kosong kepada tim Finance?**

Rating bersifat **opsional**, sehingga nilai kosong berarti "pembeli tidak memberi rating", bukan data yang hilang karena kesalahan sistem. Mengisinya dengan angka tebakan akan mendistorsi hasil:
- mengisi dengan 0 akan menurunkan rata-rata secara palsu;
- mengisi dengan nilai rata-rata akan menyempitkan variasi dan membuat data tampak lebih pasti daripada kenyataannya.

Karena itu, rating rata-rata dilaporkan **hanya dari transaksi yang memiliki rating**, disertai jumlahnya. Contoh: "Rating rata-rata [X] dari [Y] transaksi yang memberi rating ([Z]% dari 490 transaksi)." Fungsi `mean()` pada pandas secara default sudah mengabaikan NaN, sehingga perhitungan ini tidak memerlukan imputasi.